# Lab 1 — Give the KPI meeting one set of minutes

The analyst-secretary has been asked to settle a familiar office dispute: Finance wants one definition of revenue, Growth wants to slice it three different ways, and BI would like everyone to stop emailing screenshots. Build a Unity Catalog Metric View over the built-in TPC-H sample tables, then query its measures at different grains.

Run on DBR 18.1+ compute you can attach to, or a SQL warehouse where you have `CAN USE` and the required features. The definition uses YAML 1.1 metadata and `rely.at_most_one_match`; on DBR 17.3–18.0, remove the optional `rely` block. You need Unity Catalog privileges to create the schema and view, and `samples.tpch` must be available.

## 0. Choose a destination

First, pick a writable catalog and schema for the meeting minutes. This bundle uses `workspace.metric_views_lab` by default; change the names in the SQL cells if you deploy elsewhere. The schema-creation cell needs `USE CATALOG` and `CREATE SCHEMA` on the parent catalog; creating the Metric View then needs `USE SCHEMA` and `CREATE TABLE` in the destination schema. The shared `samples` catalog is the source; your Metric View belongs in your catalog.

In [ ]:
%sql
-- Bundle default: workspace.metric_views_lab. Change this if needed.
CREATE SCHEMA IF NOT EXISTS workspace.metric_views_lab;

## 1. Inspect the source grain

Before Finance accuses BI of inflating revenue, check the evidence. The `orders` table has order-header rows; the `customer` table should have one row per customer key. Confirm the counts and uniqueness assumption before modeling the lookup join.

In [ ]:
%sql
SELECT COUNT(*) AS orders, COUNT(DISTINCT o_orderkey) AS distinct_order_keys
FROM samples.tpch.orders;

In [ ]:
%sql
SELECT COUNT(*) AS customers, COUNT(DISTINCT c_custkey) AS distinct_customer_keys
FROM samples.tpch.customer;

The two counts should match. This explicit duplicate check makes the `rely.at_most_one_match` assumption fail loudly in a real dataset instead of asking someone to compare two numbers by eye. An empty result is the expected result for the sample table.

In [ ]:
%sql
SELECT c_custkey, COUNT(*) AS rows_for_key
FROM samples.tpch.customer
GROUP BY c_custkey
HAVING COUNT(*) > 1
ORDER BY rows_for_key DESC, c_custkey
LIMIT 20;

## 2. Put the definition in the minutes

The definition makes `total_revenue` explicit: it sums TPC-H order-header total prices after the model-wide date filter. AOV is composed from atomic revenue and order-count measures. The EUR format demonstrates display metadata only; TPC-H prices have no currency contract, so replace EUR with your governed unit (or use a plain number format) before real reporting. In production, document currency, tax/refund treatment, event date, and exclusions. `rely.at_most_one_match` is a promise to the optimizer; keep it only if the joined key is unique.

In [ ]:
%sql
CREATE OR REPLACE VIEW workspace.metric_views_lab.orders_metric_view
WITH METRICS LANGUAGE YAML AS
$$
version: 1.1
comment: "Order KPIs for sales analysis. Revenue is order-header total price."
source: samples.tpch.orders
joins:
  - name: customer
    source: samples.tpch.customer
    on: source.o_custkey = customer.c_custkey
    rely:
      at_most_one_match: true
filter: source.o_orderdate >= DATE'1995-01-01'
fields:
  - name: order_month
    expr: DATE_TRUNC('MONTH', source.o_orderdate)
    display_name: "Order Month"
    comment: "Calendar month of the order date."
  - name: order_status
    expr: CASE source.o_orderstatus WHEN 'O' THEN 'Open' WHEN 'P' THEN 'Partially fulfilled' WHEN 'F' THEN 'Fulfilled' END
  - name: market_segment
    expr: customer.c_mktsegment
  - name: customer_key
    expr: source.o_custkey
measures:
  - name: order_count
    expr: COUNT(1)
    display_name: "Orders"
  - name: total_revenue
    expr: SUM(source.o_totalprice)
    display_name: "Order Revenue"
    synonyms: ["sales", "order value"]
    format:
      type: currency
      currency_code: EUR
      decimal_places:
        type: exact
        places: 2
  - name: fulfilled_revenue
    expr: SUM(source.o_totalprice) FILTER (WHERE source.o_orderstatus = 'F')
    display_name: "Fulfilled Order Revenue"
  - name: average_order_value
    expr: MEASURE(total_revenue) / MEASURE(order_count)
    display_name: "Average Order Value"
    format:
      type: currency
      currency_code: EUR
      decimal_places:
        type: exact
        places: 2
$$;

## 3. Ask the same metric at two grains

Give Finance and Growth the same definition, then let each ask a different grouping question. `MEASURE(...)` evaluates a measure at the requested grain. `GROUP BY ALL` groups by every non-aggregated selected field.

In [ ]:
%sql
SELECT order_month, market_segment,
       MEASURE(total_revenue) AS revenue,
       MEASURE(order_count) AS orders,
       MEASURE(average_order_value) AS average_order_value
FROM workspace.metric_views_lab.orders_metric_view
GROUP BY ALL
ORDER BY order_month, market_segment;

In [ ]:
%sql
SELECT market_segment, MEASURE(total_revenue) AS revenue,
       MEASURE(order_count) AS orders,
       MEASURE(average_order_value) AS average_order_value
FROM workspace.metric_views_lab.orders_metric_view
GROUP BY market_segment
ORDER BY revenue DESC;

## 4. Settle the average-of-averages complaint

Finance objects: do not average precomputed segment AOVs to get an overall AOV. Segments may contain different numbers of orders. Here, the source has one row per order, so `COUNT(1)` is the atomic denominator and the composed measure divides `total_revenue` by `order_count`. A line-item source would need a distinct-order count or an order-level upstream source. Compare the result against this direct source query for each segment; investigate any mismatch before anyone forwards the chart to the CFO.

In [ ]:
%sql
SELECT c.c_mktsegment,
       SUM(o.o_totalprice) / COUNT(DISTINCT o.o_orderkey) AS direct_aov
FROM samples.tpch.orders o
LEFT JOIN samples.tpch.customer c ON o.o_custkey = c.c_custkey
WHERE o.o_orderdate >= DATE'1995-01-01'
GROUP BY c.c_mktsegment
ORDER BY c.c_mktsegment;

The two result sets above are useful to eyeball, but eyeballing is how KPI disputes get promoted. Reconcile the Metric View and direct control in one query; a non-zero `mismatches` count is a stop sign, not a rounding discussion.

In [ ]:
%sql
WITH metric AS (
  SELECT market_segment, MEASURE(average_order_value) AS metric_aov
  FROM workspace.metric_views_lab.orders_metric_view
  GROUP BY market_segment
), direct AS (
  SELECT c.c_mktsegment AS market_segment,
         SUM(o.o_totalprice) / COUNT(DISTINCT o.o_orderkey) AS direct_aov
  FROM samples.tpch.orders o
  LEFT JOIN samples.tpch.customer c ON o.o_custkey = c.c_custkey
  WHERE o.o_orderdate >= DATE'1995-01-01'
  GROUP BY c.c_mktsegment
)
SELECT COALESCE(metric.market_segment, direct.market_segment) AS market_segment,
       metric.metric_aov,
       direct.direct_aov,
       metric.metric_aov - direct.direct_aov AS delta,
       CASE WHEN metric.metric_aov IS NOT NULL
                  AND direct.direct_aov IS NOT NULL
                  AND ABS(metric.metric_aov - direct.direct_aov) < 0.000001
            THEN 'PASS' ELSE 'CHECK' END AS control_status
FROM metric
FULL OUTER JOIN direct USING (market_segment)
ORDER BY market_segment;

## 4b. Filter one measure without filtering the whole meeting

The SQL `FILTER (WHERE ...)` clause belongs inside the aggregate expression. It lets us keep every order in the model while defining a measure for one status. That is different from the top-level YAML `filter`, which removes rows for every measure. The direct control below should match the Metric View measure for fulfilled orders.

In [ ]:
%sql
SELECT order_status,
       MEASURE(total_revenue) AS all_status_revenue,
       MEASURE(fulfilled_revenue) AS fulfilled_revenue
FROM workspace.metric_views_lab.orders_metric_view
GROUP BY order_status
ORDER BY order_status;

In [ ]:
%sql
SELECT SUM(o_totalprice) FILTER (WHERE o_orderstatus = 'F') AS direct_fulfilled_revenue
FROM samples.tpch.orders
WHERE o_orderdate >= DATE'1995-01-01';

## Optional extension — let the caller choose a discount

Growth requests a 10% scenario, while Finance would prefer that nobody create six nearly identical views. A parameter supplies the caller's discount rate. This requires Databricks Runtime 18.2+ (or a SQL warehouse with the feature). Parameterized Metric Views cannot be materialized, so use this when runtime flexibility is worth that performance trade-off.

In [ ]:
%sql
CREATE OR REPLACE VIEW workspace.metric_views_lab.orders_discount_metric_view
WITH METRICS LANGUAGE YAML AS
$$
version: 1.1
source: samples.tpch.orders
parameters:
  - name: discount
    data_type: double
    default: 0
fields:
  - name: order_month
    expr: DATE_TRUNC('MONTH', source.o_orderdate)
measures:
  - name: gross_revenue
    expr: SUM(source.o_totalprice)
  - name: discounted_revenue
    expr: SUM((1 - discount) * source.o_totalprice)
$$;



Now pass a 10% discount into the same definition. Omitting the argument uses the default value of zero; the recipe stays the same, only the meeting's scenario changes.

In [ ]:
%sql
SELECT order_month, MEASURE(discounted_revenue) AS discounted_revenue
FROM workspace.metric_views_lab.orders_discount_metric_view(discount => 0.10)
GROUP BY order_month
ORDER BY order_month;

## 5. Publish and inspect the minutes

Use Catalog Explorer to inspect fields, measures, permissions, and lineage. To retrieve the saved definition for review or version control, run `DESCRIBE TABLE EXTENDED workspace.metric_views_lab.orders_metric_view AS JSON`. Consumers need `SELECT` on the Metric View plus `USE CATALOG` and `USE SCHEMA` through its parents. They do not automatically need direct `SELECT` on the source tables; the creator does need source access to define the view. For example:

```sql
GRANT SELECT ON VIEW workspace.metric_views_lab.orders_metric_view TO `data-consumers`;
```

Grant parent catalog/schema usage separately if it is not already inherited. To collaborate on edits, the owner group must be able to read the dependencies; materialized Metric Views do not support group ownership. An `ALTER VIEW` replaces the full YAML definition, so edit the complete saved definition, not a fragment. YAML 1.1 can strip single-line `#` comments on save; use the YAML `comment:` metadata for durable documentation.

### Exercises
1. Add a field for order priority and query revenue by priority.
2. Add a filtered measure for fulfilled-order revenue without filtering out other statuses from the whole model.
3. Change the model-level filter and explain which departments' reports it affects.
4. Add a precise comment defining how production revenue treats refunds, taxes, and currency conversion.

In [ ]:
%sql
DESCRIBE TABLE EXTENDED workspace.metric_views_lab.orders_metric_view AS JSON;